<a href="https://colab.research.google.com/github/anamacao/FAPESP-PIBIC-scrapping/blob/main/internetlab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://internetlab.org.br/ajax/"
SITE_URL = "https://internetlab.org.br"
print("✅ Scraper do InternetLab Blog pronto!")

✅ Scraper do InternetLab Blog pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [12]:
# %%
def load_articles_from_db():
    """Carrega os artigos salvos no banco de dados SQLite."""
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

# Recarrega os dados para garantir que pegamos o que foi coletado
df_db = load_articles_from_db()

if df_db.empty:
    print("⚠️ O banco de dados ainda está vazio. Certifique-se de executar a célula de coleta (a7b8c9d0) primeiro.")
else:
    print(f"✅ Sucesso! Total no banco: {len(df_db)} registros")
    display(df_db.head())

✅ Sucesso! Total no banco: 505 registros


,id,title,date,author,url,source
0,74,"Em ambiente de receio, confiança individual no...",31.10.2023,Ester Borges e Heloisa Massaro,https://internetlab.org.br/pt/pesquisa/em-ambi...,InternetLab
1,332,"Em meio a discussão sobre aplicativos, Lab lan...",31.10.2017,"Pedro de Paula, Beatriz Kira e Rafael Augusto ...",https://internetlab.org.br/pt/noticias/em-meio...,InternetLab
2,270,“A questão não é discutir isoladamente a soluç...,31.08.2018,Maria Luciano e Francisco Brito Cruz,https://internetlab.org.br/pt/noticias/questao...,InternetLab
3,393,Indenizações por dano moral ameaçam liberdade ...,31.08.2016,Ana Luiza Araujo,https://internetlab.org.br/pt/opiniao/indeniza...,InternetLab
4,394,"Novo Projeto Reporta: Internet, Vozes e Votos",31.08.2016,mrnvlnt,https://internetlab.org.br/pt/noticias/novo-pr...,InternetLab


In [6]:
# %%
def montar_url(pagina):
    """Monta a URL da API AJAX do InternetLab para a página indicada."""
    return f"{BASE_URL}?call=blog&lang=pt&page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página do post e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [7]:
# %%
noticias = []
TOTAL_PAGES = 150  # páginas suficientes para cobrir todo o histórico, vcs podem ajustar conforme o necessário, ok?

for pagina in range(1, TOTAL_PAGES + 1):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code} — parando.")
            break

        data = r.json()
    except Exception as e:
        print(f"⚠️ Erro ao acessar a API: {e}")
        break

    if not data.get("status") or not data.get("answer"):
        print("🏁 Sem mais resultados — fim da coleta.")
        break

    posts = data["answer"]
    print(f"   {len(posts)} posts encontrados")

    for post in posts:
        titulo = post.get("title", "")
        link = post.get("url", "")
        data_fmt = post.get("date_formatted", "NA")
        categoria = post.get("category", "")
        area = post.get("areas_pesquisa", "")
        autores = post.get("autores", "InternetLab")

        # extrai parágrafos da página do post
        paragrafos = extrair_paragrafos(link)

        # acumula
        noticias.append({
            "titulo": titulo,
            "data": data_fmt,
            "link": link,
            "categoria": categoria,
            "area": area,
            "autores": autores,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "InternetLab"
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=data_fmt,
            author=autores,
            url=link,
            source="InternetLab"
        )

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} posts")

df_internetlab = pd.DataFrame(noticias)
display(df_internetlab.head())

📄 Coletando página 1: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=1
   6 posts encontrados
📄 Coletando página 2: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=2
   6 posts encontrados
📄 Coletando página 3: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=3
   6 posts encontrados
📄 Coletando página 4: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=4
   6 posts encontrados
📄 Coletando página 5: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=5
   6 posts encontrados
📄 Coletando página 6: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=6
   6 posts encontrados
📄 Coletando página 7: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=7
   6 posts encontrados
📄 Coletando página 8: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=8
   6 posts encontrados
📄 Coletando página 9: https://internetlab.org.br/ajax/?call=blog&lang=pt&page=9
   6 posts encontrados
📄 Coletando página 10: https://internetlab.org.br/ajax/?call=blog&lang=pt

,titulo,data,link,categoria,area,autores,paragrafos,fonte
0,Como enfrentar a misoginia na internet? Intern...,08.04.2026,https://internetlab.org.br/pt/noticias/como-en...,Notícias,Desigualdades e Identidades,Clarice Tavares,Nosso boletim semanal de atualização sobre pol...,InternetLab
1,Um guia da dieta de mídia digital brasileira,07.04.2026,https://internetlab.org.br/pt/pesquisa/um-guia...,Notícias,Informação e Política,"Isabella Ferrarese, Stephanie Lima e Vitor San...",Nosso boletim semanal de atualização sobre pol...,InternetLab
2,Novo projeto do InternetLab aborda governança ...,25.03.2026,https://internetlab.org.br/pt/pesquisa/novo-pr...,Notícias,Desigualdades e Identidades,"Helena Secaf, Danyelle Reis e Fernanda Campagn...",Nosso boletim semanal de atualização sobre pol...,InternetLab
3,InternetLab apresenta sugestões ao TSE na minu...,12.02.2026,https://internetlab.org.br/pt/noticias/interne...,Notícias,Informação e Política,Camila Akemi Tsuzuki,Nosso boletim semanal de atualização sobre pol...,InternetLab
4,Falta de transparência e diretrizes nacionais ...,02.02.2026,https://internetlab.org.br/pt/noticias/falta-d...,Notícias,Privacidade e Vigilância,"Helena Secaf, Danielle Bello e Danyelle Reis",Nosso boletim semanal de atualização sobre pol...,InternetLab


In [8]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 505 registros


,id,title,date,author,url,source
0,74,"Em ambiente de receio, confiança individual no...",31.10.2023,Ester Borges e Heloisa Massaro,https://internetlab.org.br/pt/pesquisa/em-ambi...,InternetLab
1,332,"Em meio a discussão sobre aplicativos, Lab lan...",31.10.2017,"Pedro de Paula, Beatriz Kira e Rafael Augusto ...",https://internetlab.org.br/pt/noticias/em-meio...,InternetLab
2,270,“A questão não é discutir isoladamente a soluç...,31.08.2018,Maria Luciano e Francisco Brito Cruz,https://internetlab.org.br/pt/noticias/questao...,InternetLab
3,393,Indenizações por dano moral ameaçam liberdade ...,31.08.2016,Ana Luiza Araujo,https://internetlab.org.br/pt/opiniao/indeniza...,InternetLab
4,394,"Novo Projeto Reporta: Internet, Vozes e Votos",31.08.2016,mrnvlnt,https://internetlab.org.br/pt/noticias/novo-pr...,InternetLab
5,129,Conhecimento livre e as barreiras encontradas ...,31.03.2022,Fernanda K. Martins e Stephanie Lima,https://internetlab.org.br/pt/noticias/conheci...,InternetLab
6,370,[#2][especial] europa e esquecimento: desafios...,31.01.2017,Clarice Tambelli,https://internetlab.org.br/pt/noticias/2especi...,InternetLab
7,324,Lançamento do livro &#8220;A Internet no Banco...,30.11.2017,Juliana Ruiz,https://internetlab.org.br/pt/conjuntura/lanca...,InternetLab
8,325,"Black Mirror: “The Waldo Moment” (S02 E03), mí...",30.11.2017,Thiago Oliva,https://internetlab.org.br/pt/opiniao/black-mi...,InternetLab
9,183,MonitorA: discurso de ódio contra candidatas n...,30.10.2020,"Fernanda K. Martins, Alessandra Gomes, Mariana...",https://internetlab.org.br/pt/noticias/interne...,InternetLab


In [9]:
keywords = ['digital', 'internet', 'IA', 'tecnologia', 'dados', 'privacidade']
pattern = r'|'.join(keywords)

df_filt = df_internetlab[
    df_internetlab['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_internetlab['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} posts filtrados (de {len(df_internetlab)})")
display(df_filt.head())

505 posts filtrados (de 505)


,titulo,data,link,categoria,area,autores,paragrafos,fonte
0,Como enfrentar a misoginia na internet? Intern...,08.04.2026,https://internetlab.org.br/pt/noticias/como-en...,Notícias,Desigualdades e Identidades,Clarice Tavares,Nosso boletim semanal de atualização sobre pol...,InternetLab
1,Um guia da dieta de mídia digital brasileira,07.04.2026,https://internetlab.org.br/pt/pesquisa/um-guia...,Notícias,Informação e Política,"Isabella Ferrarese, Stephanie Lima e Vitor San...",Nosso boletim semanal de atualização sobre pol...,InternetLab
2,Novo projeto do InternetLab aborda governança ...,25.03.2026,https://internetlab.org.br/pt/pesquisa/novo-pr...,Notícias,Desigualdades e Identidades,"Helena Secaf, Danyelle Reis e Fernanda Campagn...",Nosso boletim semanal de atualização sobre pol...,InternetLab
3,InternetLab apresenta sugestões ao TSE na minu...,12.02.2026,https://internetlab.org.br/pt/noticias/interne...,Notícias,Informação e Política,Camila Akemi Tsuzuki,Nosso boletim semanal de atualização sobre pol...,InternetLab
4,Falta de transparência e diretrizes nacionais ...,02.02.2026,https://internetlab.org.br/pt/noticias/falta-d...,Notícias,Privacidade e Vigilância,"Helena Secaf, Danielle Bello e Danyelle Reis",Nosso boletim semanal de atualização sobre pol...,InternetLab


In [13]:
# Contagem de artigos por categoria
category_counts = df_filt['categoria'].value_counts().reset_index()
category_counts.columns = ['Categoria', 'Quantidade']

print("Contagem de artigos por categoria:")
display(category_counts)

Contagem de artigos por categoria:


,Categoria,Quantidade
0,Notícias,321
1,Opinião,65
2,InternetLab Reporta,33
3,Conjuntura,26
4,Especial,18
5,Antivírus,14
6,Pesquisa,10
7,Imprensa,8
8,Agenda,5
9,,3


In [16]:
import plotly.io as pio
pio.renderers.default = "colab"

fig_cat = px.bar(
    category_counts,
    x='Categoria',
    y='Quantidade',
    title='Distribuição de Artigos por Categoria',
    color='Quantidade',
    text_auto=True
)
fig_cat.update_layout(xaxis_tickangle=-45)
fig_cat.show()

In [17]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "colab"

# Garantir que a coluna de data est no formato datetime
df_filt['data_dt'] = pd.to_datetime(df_filt['data'], format='%d.%m.%Y', errors='coerce')

# Extrair o ano
df_filt['ano'] = df_filt['data_dt'].dt.year

# Contar artigos por ano
evolution_year = df_filt.groupby('ano').size().reset_index(name='Quantidade')

# Criar o grfico de linha
fig_line = px.line(
    evolution_year,
    x='ano',
    y='Quantidade',
    title='Evoluo Temporal de Artigos (por Ano)',
    markers=True,
    labels={'ano': 'Ano', 'Quantidade': 'Nmero de Artigos'}
)

fig_line.show()

In [18]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    import plotly.io as pio
    pio.renderers.default = "colab"

    # ------------------------------
    # Top 15
    # ------------------------------
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x='rank',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Internet Governance'
    )
    fig1.update_layout(height=600)
    fig1.show()

    # ------------------------------
    # Pizza por Fonte (BANCO)
    # ------------------------------
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Nuvem de Palavras
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{4,}\b', text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Nuvem de Palavras – Títulos'
    )
    fig3.show()

In [19]:
plot_charts(df_db)

In [20]:
def analyze_authors_by_category(df):
    # Criar uma cópia e explodir a coluna de autores se houver múltiplos separados por vírgula ou similar
    # No dataset do InternetLab, 'autores' costuma ser uma string
    df_authors = df.copy()

    # Limpeza básica e separação de múltiplos autores (ajuste o separador se necessário)
    df_authors['autores_list'] = df_authors['autores'].str.split(',| e ')
    df_authors = df_authors.explode('autores_list')
    df_authors['autores_list'] = df_authors['autores_list'].str.strip()

    # Filtrar autores genéricos ou nulos
    df_authors = df_authors[~df_authors['autores_list'].isin(['InternetLab', 'NA', ''])]

    # Contagem por Categoria e Autor
    author_cat_dist = df_authors.groupby(['categoria', 'autores_list']).size().reset_index(name='count')

    # Pegar as top 5 categorias para não poluir o gráfico
    top_categories = df['categoria'].value_counts().nlargest(5).index
    author_cat_dist = author_cat_dist[author_cat_dist['categoria'].isin(top_categories)]

    # Pegar os top 10 autores no geral para a comparação
    top_authors = df_authors['autores_list'].value_counts().nlargest(10).index
    author_cat_dist = author_cat_dist[author_cat_dist['autores_list'].isin(top_authors)]

    import plotly.io as pio
    pio.renderers.default = "colab"

    fig = px.bar(
        author_cat_dist,
        x='autores_list',
        y='count',
        color='categoria',
        title='Distribuição dos Top 10 Autores por Categoria',
        labels={'autores_list': 'Autor', 'count': 'Número de Artigos', 'categoria': 'Categoria'},
        barmode='stack'
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

analyze_authors_by_category(df_filt)